# Use CBBD Scraper to get player data

### CONFIG

In [1]:
import time
import cbbd
import pandas as pd
import numpy as np
import unicodedata

In [2]:
# configs
api_key = "your_api_key"

configuration = cbbd.Configuration(
    access_token = api_key
)

# show all columns for dataframes (preference)
pd.set_option('display.max_columns', None)

In [3]:
#functions
def clean_slug(series):
    # convert to lowercase and strip whitespace
    s = series.str.lower().str.strip()

    # remove common suffixes that cause mismatches
    suffixes = [r'\bjr\b\.?', r'\bsr\b\.?', r'\biii\b', r'\bii\b', r'\biv\b']
    for suffix in suffixes:
        s = s.str.replace(suffix, '', regex=True)

    # normalize characters
    s = s.apply(lambda x: ''.join(
        c for c in unicodedata.normalize('NFD', str(x))
        if unicodedata.category(c) != 'Mn'
    ) if pd.notnull(x) else x)

    # strip out any remaining non-alphanumeric characters and spaces
    s = s.str.replace(r'[^a-z0-9]', '', regex=True)
    return s

def get_player_stats(season_year, api_config):
    """gets season statistics for all players in a year """
    with cbbd.ApiClient(api_config) as api_client:
        api_instance = cbbd.StatsApi(api_client)
        try:
            return api_instance.get_player_season_stats(season=season_year)
        except Exception as e:
            print(f"Exception when calling StatsApi->get_player_season_stats for {season_year}: {e}\n")
            return None

def get_draft_picks(api_config):
    """gets all historical draft pick profiles """
    with cbbd.ApiClient(api_config) as api_client:
        api_instance = cbbd.DraftApi(api_client)
        try:
            return api_instance.get_draft_picks()
        except Exception as e:
            print(f"Exception when calling DraftApi->get_draft_picks: {e}\n")
            return None

def get_team_rosters(season_year, api_config):
    """ gets team rosters """
    with cbbd.ApiClient(api_config) as api_client:
        api_instance = cbbd.TeamsApi(api_client)
        try:
            return api_instance.get_team_roster(season=season_year)
        except Exception as e:
            print(f"Exception when calling StatsApi->get_team_rosters for {season_year}: {e}\n")
            return None

def get_ratings(season_year, api_config):
    """ gets ratings for teams """
    with cbbd.ApiClient(api_config) as api_client:
        api_instance = cbbd.RatingsApi(api_client)
        try:
            return api_instance.get_adjusted_efficiency(season=season_year)
        except Exception as e:
            print(f"Exception when calling StatsApi->get_team_rosters for {season_year}: {e}\n")
            return None

### Player Stats API

In [4]:
# player stats api
# loop through the season stats and go until season before most recent
all_stats_records = []

print("Starting to gather season stats.")
for year in range(2007, 2026):
    print(f"Fetching data for the {year} season...")
    year_data = get_player_stats(season_year=year, api_config=configuration)

    if year_data:
        for obj in year_data:
            # generate a normalized clean string of the name for fallback matching
            name_slug = "".join(c for c in obj.name.lower() if c.isalnum()) if obj.name else ""

            flattened_obj = {
                "athlete_id": obj.athlete_id,
                "name_slug": name_slug,
                "season": obj.season,
                "season_label": obj.season_label,
                "team_id": obj.team_id,
                "team": obj.team,
                "conference": obj.conference,
                "athlete_source_id": getattr(obj, 'athlete_source_id', None),
                "name": obj.name,
                "position": obj.position,
                "games": obj.games,
                "starts": obj.starts,
                "minutes": obj.minutes,
                "points": obj.points,
                "turnovers": obj.turnovers,
                "fouls": obj.fouls,
                "assists": obj.assists,
                "steals": obj.steals,
                "blocks": obj.blocks,
                "usage": obj.usage,
                "offensive_rating": obj.offensive_rating,
                "defensive_rating": obj.defensive_rating,
                "net_rating": obj.net_rating,
                "porpag": obj.porpag,
                "effective_field_goal_pct": obj.effective_field_goal_pct,
                "true_shooting_pct": obj.true_shooting_pct,
                "assists_turnover_ratio": obj.assists_turnover_ratio,
                "free_throw_rate": obj.free_throw_rate,
                "offensive_rebound_pct": obj.offensive_rebound_pct,
                "fg_pct": obj.field_goals.pct if obj.field_goals else None,
                "fg_attempted": obj.field_goals.attempted if obj.field_goals else None,
                "fg_made": obj.field_goals.made if obj.field_goals else None,
                "two_pt_fg_pct": obj.two_point_field_goals.pct if obj.two_point_field_goals else None,
                "two_pt_fg_attempted": obj.two_point_field_goals.attempted if obj.two_point_field_goals else None,
                "two_pt_fg_made": obj.two_point_field_goals.made if obj.two_point_field_goals else None,
                "three_pt_fg_pct": obj.three_point_field_goals.pct if obj.three_point_field_goals else None,
                "three_pt_fg_attempted": obj.three_point_field_goals.attempted if obj.three_point_field_goals else None,
                "three_pt_fg_made": obj.three_point_field_goals.made if obj.three_point_field_goals else None,
                "ft_pct": obj.free_throws.pct if obj.free_throws else None,
                "ft_attempted": obj.free_throws.attempted if obj.free_throws else None,
                "ft_made": obj.free_throws.made if obj.free_throws else None,
                "rebounds_total": obj.rebounds.total if obj.rebounds else None,
                "rebounds_defensive": obj.rebounds.defensive if obj.rebounds else None,
                "rebounds_offensive": obj.rebounds.offensive if obj.rebounds else None,
                "win_shares_total_per40": obj.win_shares.total_per40 if obj.win_shares else None,
                "win_shares_total": obj.win_shares.total if obj.win_shares else None,
                "win_shares_defensive": obj.win_shares.defensive if obj.win_shares else None,
                "win_shares_offensive": obj.win_shares.offensive if obj.win_shares else None,
            }
            all_stats_records.append(flattened_obj)
        print(f"Processed {len(year_data)} players for {year}.\n")
    else:
        print(f"No data returned or error occurred for the {year} season.\n")

    time.sleep(1.5)

df_stats = pd.DataFrame(all_stats_records)

Starting to gather season stats.
Fetching data for the 2007 season...
Successfully processed 6429 players for 2007.

Fetching data for the 2008 season...
Successfully processed 6786 players for 2008.

Fetching data for the 2009 season...
Successfully processed 5242 players for 2009.

Fetching data for the 2010 season...
Successfully processed 5489 players for 2010.

Fetching data for the 2011 season...
Successfully processed 5401 players for 2011.

Fetching data for the 2012 season...
Successfully processed 5387 players for 2012.

Fetching data for the 2013 season...
Successfully processed 5656 players for 2013.

Fetching data for the 2014 season...
Successfully processed 8241 players for 2014.

Fetching data for the 2015 season...
Successfully processed 8360 players for 2015.

Fetching data for the 2016 season...
Successfully processed 8603 players for 2016.

Fetching data for the 2017 season...
Successfully processed 8729 players for 2017.

Fetching data for the 2018 season...
Succes

In [5]:
# keep the most recent year for each player
# sort by season ascending so the most recent year is at the bottom
df_stats = df_stats.sort_values(by="season", ascending=True)

# drop duplicates by athlete_id, keeping the last (most recent) record
df_stats_latest = df_stats.drop_duplicates(subset=["athlete_id"], keep="last").copy()

### Team Stats API (get rosters)

In [7]:
# team stats api
# loop through the team rosters and go until season before most recent
all_roster_records = []

print("Starting to gather team rosters.")
for year in range(2007, 2026):
    print(f"Fetching data for the {year} season...")
    year_data = get_team_rosters(season_year=year, api_config=configuration)

    if year_data:
        for obj in year_data:
            # extract all top-level attributes from the object
            try:
                obj_dict = vars(obj).copy()
            except TypeError:
                obj_dict = getattr(obj, '__dict__', {}).copy()

            # handle nested objects dynamically
            flattened_obj = {}
            for key, val in obj_dict.items():
                # if the value has its own attributes (nested object) and isn't a string/list/dict
                if val and not isinstance(val, (str, int, float, list, dict, bool)):
                    try:
                        nested_dict = vars(val)
                        for n_key, n_val in nested_dict.items():
                            flattened_obj[f"{key}_{n_key}"] = n_val
                    except TypeError:
                        # fallback if the nested property isn't an object with a dictionary
                        flattened_obj[key] = val
                else:
                    flattened_obj[key] = val

            # handle fallback name_slug dynamically if 'name' is in the attributes
            if "name" in flattened_obj and flattened_obj["name"]:
                flattened_obj["name_slug"] = "".join(c for c in str(flattened_obj["name"]).lower() if c.isalnum())
            else:
                flattened_obj["name_slug"] = ""

            all_roster_records.append(flattened_obj)

        print(f"Processed {len(year_data)} players for {year}.\n")
    else:
        print(f"No data returned or error occurred for the {year} season.\n")

    time.sleep(1.5)

# convert to df
df_rosters = pd.DataFrame(all_roster_records)

Starting to gather team rosters.
Fetching data for the 2007 season...
Successfully processed 1518 players for 2007.

Fetching data for the 2008 season...
Successfully processed 1518 players for 2008.

Fetching data for the 2009 season...
Successfully processed 1518 players for 2009.

Fetching data for the 2010 season...
Successfully processed 1518 players for 2010.

Fetching data for the 2011 season...
Successfully processed 1518 players for 2011.

Fetching data for the 2012 season...
Successfully processed 1518 players for 2012.

Fetching data for the 2013 season...
Successfully processed 1518 players for 2013.

Fetching data for the 2014 season...
Successfully processed 1518 players for 2014.

Fetching data for the 2015 season...
Successfully processed 1518 players for 2015.

Fetching data for the 2016 season...
Successfully processed 1518 players for 2016.

Fetching data for the 2017 season...
Successfully processed 1518 players for 2017.

Fetching data for the 2018 season...
Succes

In [8]:
# drop teams that don't have any players listed
df_filtered = df_rosters[df_rosters['players'].map(len) > 0].copy()

# "explode" the players column so every player in the list gets their own row
df_exploded = df_filtered.explode('players')

all_parsed_rosters = []

# iterate through the exploded rows and dynamically parse the player objects
for idx, row in df_exploded.iterrows():
    obj = row['players']

    # extract the team/season metadata from the current row
    base_record = {
        "team_id": row['team_id'],
        "team_source_id": row['team_source_id'],
        "team": row['team'],
        "conference": row['conference'],
        "season": row['season']
    }

    # dynamically extract all attributes belonging to the player object
    try:
        player_dict = vars(obj).copy()
    except TypeError:
        player_dict = getattr(obj, '__dict__', {}).copy()

    # flatten any nested structures inside the player object if they exist
    flattened_player = {}
    for key, val in player_dict.items():
        if val and not isinstance(val, (str, int, float, list, dict, bool)):
            try:
                nested_dict = vars(val)
                for n_key, n_val in nested_dict.items():
                    flattened_player[f"{key}_{n_key}"] = n_val
            except TypeError:
                flattened_player[key] = val
        else:
            flattened_player[key] = val

    # add a clean name_slug for the player if they have a name attribute
    if "name" in flattened_player and flattened_player["name"]:
        flattened_player["name_slug"] = "".join(c for c in str(flattened_player["name"]).lower() if c.isalnum())
    else:
        flattened_player["name_slug"] = ""

    # combine the team metadata with the parsed player stats
    combined_record = {**base_record, **flattened_player}
    all_parsed_rosters.append(combined_record)

# turn it back into a df
df_rosters_final = pd.DataFrame(all_parsed_rosters)

In [9]:
# drop unneeded columns
df_rosters_final = df_rosters_final.drop(columns=['first_name', 'last_name', 'jersey', 'hometown_county_fips', 'hometown_longitude',
                                                 'hometown_latitude', 'hometown_country'])

# get age based off season and date of birth
# ensure date of birth column is a datetime object
df_rosters_final['date_of_birth'] = pd.to_datetime(df_rosters_final['date_of_birth'])

# subtract the birth year from the season year
df_rosters_final['age'] = df_rosters_final['season'] - df_rosters_final['date_of_birth'].dt.year

display(df_rosters_final)

,team_id,team_source_id,team,conference,season,id,source_id,name,position,height,weight,hometown_state,hometown_city,date_of_birth,start_season,end_season,name_slug,age
0,96,45,George Washington,A-10,2007,76226,31994,Travis King,Guard,74.0,215.0,CT,New Haven,1986-01-16,2007,2010,travisking,21.0
1,96,45,George Washington,A-10,2007,78453,33561,Johnny Lee,Guard,69.0,165.0,TN,Nashville,1986-12-02,2007,2009,johnnylee,21.0
2,96,45,George Washington,A-10,2007,78597,31997,Damian Hollis,Forward,80.0,215.0,FL,Fort Lauderdale,1988-08-19,2007,2010,damianhollis,19.0
3,96,45,George Washington,A-10,2007,78598,31996,Hermann Opoku,Forward,82.0,236.0,Austria,Vienna,1986-03-07,2007,2010,hermannopoku,21.0
4,96,45,George Washington,A-10,2007,79700,27228,Rob Diggs,Forward,80.0,202.0,MD,Brandywine,NaT,2006,2009,robdiggs,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192825,616,2985,Concordia-Ann Arbor,NaN,2025,9314,4907749,Tarek Abdel-Kireem,Athlete,NaN,NaN,NaN,NaN,NaT,2023,2025,tarekabdelkireem,NaN
192826,616,2985,Concordia-Ann Arbor,NaN,2025,9315,4907748,Abeal Ayalew,Athlete,NaN,NaN,NaN,NaN,NaT,2022,2025,abealayalew,NaN
192827,616,2985,Concordia-Ann Arbor,NaN,2025,9316,4907745,Matt Kohler,Athlete,NaN,NaN,NaN,NaN,NaT,2022,2025,mattkohler,NaN
192828,616,2985,Concordia-Ann Arbor,NaN,2025,9317,4907744,Devyn Jones,Athlete,NaN,NaN,NaN,NaN,NaT,2022,2025,devynjones,NaN


In [10]:
# merge roster and player stats dfs
# create temp dfs
df_stats_merge = df_stats_latest
df_roster_merge = df_rosters_final

# choose which columns to keep from roster
roster_cols_to_keep = ['name_slug', 'season', 'team', 'hometown_state', 'hometown_city', 'date_of_birth', 'start_season', 'end_season',
                       'age']

# merge on 'name_slug', 'season', and 'team' to keep unique
df_stats_roster_merge = pd.merge(
    df_stats_merge,
    df_roster_merge[roster_cols_to_keep],
    on=['name_slug', 'season', 'team'],
    how='left' # Or 'inner', depending on your goal
)

In [11]:
display(df_stats_roster_merge)

,athlete_id,name_slug,season,season_label,team_id,team,conference,athlete_source_id,name,position,games,starts,minutes,points,turnovers,fouls,assists,steals,blocks,usage,offensive_rating,defensive_rating,net_rating,porpag,effective_field_goal_pct,true_shooting_pct,assists_turnover_ratio,free_throw_rate,offensive_rebound_pct,fg_pct,fg_attempted,fg_made,two_pt_fg_pct,two_pt_fg_attempted,two_pt_fg_made,three_pt_fg_pct,three_pt_fg_attempted,three_pt_fg_made,ft_pct,ft_attempted,ft_made,rebounds_total,rebounds_defensive,rebounds_offensive,win_shares_total_per40,win_shares_total,win_shares_defensive,win_shares_offensive,hometown_state,hometown_city,date_of_birth,start_season,end_season,age
0,86129,kylevisser,2007,20062007,342,Wake Forest,ACC,15177,Kyle Visser,C,29,20,850,497,58,78,16,20,41,26.6,117.3,141.1,-23.8,3.7,57.7,0.615,0.28,73.8,39.4,57.7,305,176,58.1,303,176,0.0,2,0,64.4,225,145,221,134,87,0.108,2.3,-1.8,4.1,MI,Grand Rapids,2003-09-17,2004.0,2007.0,4.0
1,86128,michaeldrum,2007,20062007,342,Wake Forest,ACC,26005,Michael Drum,F,28,6,652,239,39,81,31,23,0,16.9,118.0,149.3,-31.3,1.9,57.1,0.639,0.79,48.7,28.6,46.1,154,71,44.0,84,37,48.6,70,34,84.0,75,63,70,50,20,0.012,0.2,-1.9,2.1,NC,Rural Hall,NaT,2006.0,2007.0,NaN
2,86127,kevinswinton,2007,20062007,342,Wake Forest,ACC,27231,Kevin Swinton,F,25,2,221,68,24,29,0,5,4,18.7,85.0,146.8,-61.8,-0.1,57.4,0.557,0.00,68.1,48.1,57.4,47,27,57.4,47,27,0.0,0,0,43.8,32,14,54,28,26,-0.145,-0.8,-0.6,-0.2,NC,Greensboro,NaT,2006.0,2007.0,NaN
3,85334,ryancallahan,2007,20062007,341,Wagner,NEC,32862,Ryan Callahan,G,17,0,63,31,4,9,2,3,0,26.8,104.6,1431.9,-1327.3,0.3,42.0,0.505,0.50,52.0,66.7,40.0,25,10,42.9,21,9,25.0,4,1,76.9,13,10,9,3,6,-3.619,-5.7,-7.7,2.0,NH,Contoocook,NaT,2007.0,2007.0,NaN
4,85333,christianmartinusnielsen,2007,20062007,341,Wagner,NEC,32863,Christian Martinus Nielsen,G-F,6,0,40,8,2,3,0,0,0,11.0,100.5,NaN,NaN,0.2,50.0,0.592,0.00,80.0,40.0,40.0,5,2,50.0,2,1,33.3,3,1,75.0,4,3,5,3,2,NaN,NaN,NaN,NaN,WI,Denmark,NaT,2007.0,2007.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75291,7310,jaylinhenderson,2025,20242025,231,Portland State,Big Sky,5175974,Jaylin Henderson,G,32,30,1038,415,63,64,55,41,19,21.4,104.5,101.5,3.0,1.9,51.1,0.534,0.87,22.0,14.6,43.2,354,153,52.7,184,97,32.9,170,56,67.9,78,53,123,105,18,0.108,2.8,1.6,1.2,KS,Wichita,NaT,2025.0,2026.0,NaN
75292,7311,qiantmyers,2025,20242025,231,Portland State,Big Sky,5118611,Qiant Myers,G,32,32,1058,344,113,63,196,31,20,21.3,97.4,104.4,-7.0,1.1,46.9,0.499,1.73,39.1,13.6,41.8,294,123,46.7,199,93,31.6,95,30,59.1,115,68,81,70,11,0.076,2.0,1.4,0.6,CA,Fresno,NaT,2025.0,2025.0,NaN
75293,7312,shanenowell,2025,20242025,231,Portland State,Big Sky,4433263,Shane Nowell,G,30,15,481,185,27,57,22,25,9,21.0,104.9,98.1,6.8,0.9,47.1,0.518,0.81,36.4,22.7,40.9,154,63,46.8,94,44,31.7,60,19,71.4,56,40,97,75,22,0.125,1.5,0.9,0.6,WA,Seattle,NaT,2025.0,2025.0,NaN
75294,8431,vincentdelano,2025,20242025,230,Portland,WCC,5176494,Vincent Delano,G,32,16,738,168,52,43,65,30,6,17.3,83.5,121.2,-37.7,-0.3,40.2,0.432,1.25,20.8,20.5,35.4,178,63,41.4,111,46,25.4,67,17,67.6,37,25,73,58,15,-0.038,-0.7,-0.2,-0.5,AZ,Phoenix,NaT,2024.0,2025.0,NaN


### Draft API (get all draft picks)

In [17]:
# draft api
draft_response = get_draft_picks(api_config=configuration)

flattened_draft_list = []
if draft_response:
    for obj in draft_response:
        name_slug = "".join(c for c in obj.name.lower() if c.isalnum()) if obj.name else ""

        flattened_draft_obj = {
            "athlete_id": obj.athlete_id,
            "name_slug": name_slug,
            "name": obj.name,
            "height_inches": obj.height,
            "weight_lbs": obj.weight,
            "draft_year": obj.year,
            "draft_round": obj.round,
            "overall_pick": obj.overall,
            "source_team_location": obj.source_team_location
        }
        flattened_draft_list.append(flattened_draft_obj)

df_draft = pd.DataFrame(flattened_draft_list)

Fetching NBA Draft entries...


In [18]:
display(df_draft)

,athlete_id,name_slug,name,height_inches,weight_lbs,draft_year,draft_round,overall_pick,source_team_location
0,205.0,cooperflagg,Cooper Flagg,80.0,221.0,2025,1,1,Duke
1,7871.0,dylanharper,Dylan Harper,77.0,213.0,2025,1,2,Rutgers
2,158.0,vjedgecombe,VJ Edgecombe,76.0,193.0,2025,1,3,Baylor
3,206.0,konknueppel,Kon Knueppel,77.0,219.0,2025,1,4,Duke
4,7882.0,acebailey,Ace Bailey,80.0,203.0,2025,1,5,Rutgers
...,...,...,...,...,...,...,...,...,...
1479,NaN,kennysatterfield,Kenny Satterfield,74.0,185.0,2001,2,54,Cincinnati
1480,92649.0,mauricejeffers,Maurice Jeffers,76.0,195.0,2001,2,55,Saint Louis
1481,NaN,robertasjavtokas,Robertas Javtokas,82.0,220.0,2001,2,56,Lithuania
1482,NaN,alvinjones,Alvin Jones,83.0,265.0,2001,2,57,Georgia Tech


In [19]:
# merge roster and player stats dfs
# create temp dfs
temp_df_stats_roster_merge = df_stats_roster_merge
df_draft_merge = df_draft

# choose which columns to keep from draft
draft_cols_to_keep = ['name_slug', 'height_inches', 'weight_lbs', 'draft_year', 'draft_round', 'overall_pick', 'source_team_location']

# merge on 'name_slug', 'season', and 'team' (and variations) to keep unique
df_merged = pd.merge(
    temp_df_stats_roster_merge,
    df_draft_merge[draft_cols_to_keep],
    left_on=['name_slug', 'season', 'team'],
    right_on=['name_slug', 'draft_year', 'source_team_location'],
    how='left'
)

In [20]:
# keep only rows that have been drafted and have information
df_merged_filter = df_merged.dropna()

display(df_merged_filter)

,athlete_id,name_slug,season,season_label,team_id,team,conference,athlete_source_id,name,position,games,starts,minutes,points,turnovers,fouls,assists,steals,blocks,usage,offensive_rating,defensive_rating,net_rating,porpag,effective_field_goal_pct,true_shooting_pct,assists_turnover_ratio,free_throw_rate,offensive_rebound_pct,fg_pct,fg_attempted,fg_made,two_pt_fg_pct,two_pt_fg_attempted,two_pt_fg_made,three_pt_fg_pct,three_pt_fg_attempted,three_pt_fg_made,ft_pct,ft_attempted,ft_made,rebounds_total,rebounds_defensive,rebounds_offensive,win_shares_total_per40,win_shares_total,win_shares_defensive,win_shares_offensive,hometown_state,hometown_city,date_of_birth,start_season,end_season,age,height_inches,weight_lbs,draft_year,draft_round,overall_pick,source_team_location
11,86485,spencerhawes,2007,20062007,343,Washington,Pac-12,31657,Spencer Hawes,C,31,23,896,461,78,70,60,16,54,25.7,111.3,116.2,-4.9,2.8,53.3,0.568,0.77,27.0,31.0,53.2,363,193,53.3,360,192,33.3,3,1,75.5,98,74,197,136,61,0.121,2.7,0.2,2.5,WA,Seattle,1988-04-28,2007.0,2007.0,19.0,85.0,244.0,2007.0,1.0,10.0,Washington
92,86985,gabepruitt,2007,20062007,323,USC,Pac-12,22161,Gabe Pruitt,G,26,19,861,324,48,56,113,47,7,21.3,115.4,131.2,-15.8,3.1,51.2,0.564,2.35,34.0,20.5,41.6,250,104,49.6,113,56,35.0,137,48,80.0,85,68,73,58,15,0.093,2.0,-1.1,3.1,CA,Los Angeles,2004-10-07,2005.0,2007.0,3.0,76.0,170.0,2007.0,2.0,32.0,USC
98,86979,nickyoung,2007,20062007,323,USC,Pac-12,22162,Nick Young,G-F,36,23,1201,630,88,90,48,25,10,26.4,113.1,143.0,-29.9,3.6,57.1,0.614,0.55,35.4,25.4,52.3,444,232,54.3,348,189,44.8,96,43,78.3,157,123,169,126,43,0.083,2.5,-2.8,5.3,CA,Los Angeles,1985-06-01,2005.0,2007.0,22.0,79.0,206.0,2007.0,1.0,16.0,USC
113,86806,derrickbyars,2007,20062007,336,Vanderbilt,SEC,11772,Derrick Byars,G-F,31,14,990,531,73,73,108,43,4,28.1,112.5,203.5,-91.0,3.6,54.2,0.570,1.48,27.7,32.9,45.5,415,189,52.9,221,117,37.1,194,72,70.4,115,81,152,102,50,-0.032,-0.8,-7.9,7.1,TN,Memphis,1984-04-25,2006.0,2007.0,23.0,79.0,220.0,2007.0,2.0,42.0,Vanderbilt
297,86462,alandotucker,2007,20062007,355,Wisconsin,Big Ten,12288,Alando Tucker,F,35,26,1144,687,58,45,73,31,10,32.7,115.3,117.9,-2.6,4.8,50.2,0.543,1.26,46.2,39.4,46.6,526,245,51.0,406,207,31.7,120,38,65.4,243,159,188,114,74,0.199,5.7,0.0,5.7,IL,Lockport,1984-02-11,2003.0,2007.0,23.0,77.0,205.0,2007.0,1.0,29.0,Wisconsin
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65047,15376,harrisoningram,2024,20232024,200,North Carolina,ACC,4433618,Harrison Ingram,F,37,36,1213,451,51,71,80,51,15,18.9,114.4,97.1,17.3,2.7,51.6,0.534,1.57,25.9,29.7,43.0,379,163,46.7,210,98,38.5,169,65,61.2,98,60,327,230,97,0.158,4.8,2.4,2.4,TX,Dallas,2002-11-27,2024.0,2024.0,22.0,77.0,234.0,2024.0,2.0,48.0,North Carolina
65332,17024,jonathanmogbo,2024,20232024,259,San Francisco,WCC,5107897,Jonathan Mogbo,F,34,34,984,484,60,89,124,54,28,22.6,131.1,91.0,40.1,4.6,63.6,0.654,2.07,32.1,32.8,63.6,324,206,64.0,322,206,0.0,2,0,69.2,104,72,345,232,113,0.276,6.8,2.5,4.3,FL,West Palm Beach,2001-10-29,2024.0,2024.0,23.0,78.0,217.0,2024.0,2.0,31.0,San Francisco
65404,16084,bubcarrington,2024,20232024,229,Pittsburgh,ACC,4845374,Bub Carrington,G,33,33,1097,456,64,78,136,19,8,23.0,111.9,104.3,7.6,3.0,49.6,0.534,2.13,24.1,9.4,41.2,386,159,51.1,184,94,32.2,202,65,78.5,93,73,171,155,16,0.135,3.7,1.4,2.3,MD,Baltimore,2005-07-21,2024.0,2024.0,19.0,76.0,195.0,2024.0,1.0,14.0,Pittsburgh
65477,15234,zachedey,2024,20232024,236,Purdue,Big Ten,4600663,Zach Edey,C,39,39,1248,983,90,76,79,11,84,33.1,134.0,96.6,37.4,8.0,62.4,0.673,0.88,80.9,38.2,62.3,539,336,62.4,537,335,50.0,2,1,71.1,436,310,474,293,181,0.330,10.3,2.5,7.8,ON,Toronto,2002-03-14,2021.0,2024.0,22.0,88.0,299.0,2024.0,1.0,9.0,Purdue


In [21]:
# create a csv to use later or have
df_merged_filter.to_csv('all_college_data.csv', index=False)

# USE NBA API TO GET NBA METRIC

In [22]:
# scrape nba advanced stats
all_seasons_list = []

for year in range(2001, 2027):
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"
    print(f"Fetching Advanced Table for the {year} season... ", end="")

    try:
        tables = pd.read_html(url)
        df_season = tables[0]

        # clean up multi-indexed headers if they exist
        if isinstance(df_season.columns, pd.MultiIndex):
            df_season.columns = df_season.columns.get_level_values(-1)

        df_season.columns = df_season.columns.str.strip()

        # filter out repeating header rows from pagination
        df_season = df_season[df_season['Player'] != 'Player']

        # keep track of the year
        df_season['Season_Year'] = year
        all_seasons_list.append(df_season)
        print("Success")

        # delay to stay under the 20 requests/min limit
        time.sleep(3.5)

    except Exception as e:
        print(f"Failed to fetch year {year}: {e}")
        continue

# combine all individual seasons into one df
df_nba_all_time = pd.concat(all_seasons_list, ignore_index=True)

Gathering seasonal advanced stats from 2001 through 2026...
Fetching Advanced Table for the 2001 season... Success.
Fetching Advanced Table for the 2002 season... Success.
Fetching Advanced Table for the 2003 season... Success.
Fetching Advanced Table for the 2004 season... Success.
Fetching Advanced Table for the 2005 season... Success.
Fetching Advanced Table for the 2006 season... Success.
Fetching Advanced Table for the 2007 season... Success.
Fetching Advanced Table for the 2008 season... Success.
Fetching Advanced Table for the 2009 season... Success.
Fetching Advanced Table for the 2010 season... Success.
Fetching Advanced Table for the 2011 season... Success.
Fetching Advanced Table for the 2012 season... Success.
Fetching Advanced Table for the 2013 season... Success.
Fetching Advanced Table for the 2014 season... Success.
Fetching Advanced Table for the 2015 season... Success.
Fetching Advanced Table for the 2016 season... Success.
Fetching Advanced Table for the 2017 season.

In [23]:
display(df_nba_all_time)

,Rk,Player,Age,Team,Pos,G,GS,MP,PER,TS%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Awards,Season_Year
0,1.0,Michael Finley,27.0,DAL,SF,82.0,82.0,3443.0,18.2,0.521,0.169,0.209,3.6,10.0,6.9,18.3,1.8,0.7,10.1,24.9,5.6,2.9,8.5,0.119,2.4,-0.5,1.9,3.4,"MVP-15,AS",2001
1,2.0,Antoine Walker,24.0,BOS,PF,81.0,81.0,3396.0,18.9,0.505,0.351,0.202,5.0,20.4,12.3,26.8,2.1,1.1,13.8,29.1,2.7,3.8,6.6,0.093,3.5,-0.5,3.0,4.3,NaN,2001
2,3.0,Antawn Jamison,24.0,GSW,SF,82.0,82.0,3394.0,19.0,0.499,0.113,0.295,8.1,14.8,11.2,9.5,1.7,0.6,8.9,27.9,5.4,1.3,6.7,0.094,2.8,-2.0,0.8,2.3,NaN,2001
3,4.0,Anthony Mason,34.0,MIA,PF,80.0,80.0,3254.0,17.4,0.555,0.000,0.497,6.2,21.9,14.1,14.2,1.4,0.6,13.3,20.0,6.0,5.6,11.6,0.171,0.8,1.1,1.9,3.2,"MVP-15,DPOY-8,AS",2001
4,5.0,Gary Payton,32.0,SEA,PG,79.0,79.0,3244.0,22.1,0.522,0.171,0.223,2.6,10.1,6.4,36.6,2.1,0.6,10.7,27.2,9.1,1.7,10.8,0.160,4.7,-0.7,4.0,4.9,"AS,NBA3,DEF1",2001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16151,579.0,Colby Jones,23.0,DET,SG,1.0,0.0,7.0,12.9,0.333,0.667,0.000,0.0,63.1,32.1,37.8,0.0,0.0,0.0,17.8,0.0,0.0,0.0,0.124,-3.2,5.7,2.4,0.0,NaN,2026
16152,580.0,Noa Essengue,19.0,CHI,PF,2.0,0.0,6.0,-11.9,0.000,0.667,0.000,0.0,0.0,0.0,0.0,7.8,0.0,0.0,20.8,-0.1,0.0,-0.1,-0.417,-20.7,-3.0,-23.7,0.0,NaN,2026
16153,581.0,Trentyn Flowers,20.0,CHI,SF,2.0,0.0,6.0,16.7,0.667,0.333,0.000,17.9,0.0,8.8,30.5,0.0,0.0,25.0,27.8,0.0,0.0,0.0,0.023,1.7,-2.7,-1.0,0.0,NaN,2026
16154,582.0,Darius Brown II,26.0,CLE,SG,1.0,0.0,3.0,-7.7,0.000,0.000,0.000,0.0,36.5,18.3,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,-0.269,-16.3,-2.4,-18.7,0.0,NaN,2026


In [43]:
# clean up player names (remove active markers and whitespace)
df_nba_all_time['Player_Clean'] = df_nba_all_time['Player'].str.replace('*', '', regex=False).str.strip()

# choose columns and convert to numeric
numeric_cols = ['G', 'MP', 'PER', 'BPM', 'VORP', 'WS', 'WS/48']
for col in numeric_cols:
    df_nba_all_time[col] = pd.to_numeric(df_nba_all_time[col], errors='coerce').fillna(0)

# if a player was traded or added over the season, Basketball-Reference creates a '2TM' or '3TM' row
# sort to keep 'TM' rows first and drop the specific team segments so we don't double count stats.
df_nba_all_time = df_nba_all_time.sort_values(by=['Season_Year', 'Player_Clean', 'Team'])
df_nba_all_time['is_tm'] = df_nba_all_time['Team'].isin(['2TM', '3TM'])
df_nba_all_time = df_nba_all_time.sort_values(by=['Season_Year', 'Player_Clean', 'is_tm'], ascending=[True, True, False])
df_nba_all_time = df_nba_all_time.drop_duplicates(subset=['Season_Year', 'Player_Clean'], keep='first')

# filter out individual seasons where the player played less than 5 games
df_nba_all_time = df_nba_all_time[df_nba_all_time['G'] > 5]

# aggregate all results
df_career = df_nba_all_time.groupby('Player_Clean').agg(
    total_games=('G', 'sum'),
    total_minutes=('MP', 'sum'),
    total_vorp=('VORP', 'sum'),
    total_ws=('WS', 'sum'),
    career_bpm=('BPM', 'mean'),
    seasons_played=('Season_Year', 'count')
).reset_index()

# calculate their true average minutes per season across their career
df_career['avg_mp_per_season'] = df_career['total_minutes'] / df_career['seasons_played']

# filter out players who averaged 500 minutes or fewer per season
df_career = df_career[df_career['avg_mp_per_season'] > 500]

# clean up the temporary helper columns not needed anymore
df_career = df_career.drop(columns=['seasons_played', 'avg_mp_per_season'])

# create the clean match slug
df_career['match_slug'] = clean_slug(df_career['Player_Clean'])

In [38]:
display(df_career)

,Player_Clean,total_games,total_minutes,total_vorp,total_ws,career_bpm,match_slug
0,A.C. Green,82.0,1411.0,-0.1,3.2,-2.200000,acgreen
1,A.J. Green,242.0,4888.0,-0.8,7.1,-2.125000,ajgreen
5,A.J. Price,261.0,3929.0,0.6,4.7,-1.866667,ajprice
6,AJ Griffin,92.0,1572.0,0.1,1.8,-5.200000,ajgriffin
7,AJ Johnson,77.0,1093.0,-1.8,-1.3,-8.500000,ajjohnson
...,...,...,...,...,...,...,...
2572,Ömer Aşık,471.0,9216.0,0.2,20.8,-2.787500,omerask
2573,Šarūnas Jasikevičius,138.0,2530.0,0.8,4.4,-1.200000,sarunasjasikevicius
2574,Žan Tabak,55.0,777.0,-0.4,0.9,-4.200000,zantabak
2575,Žarko Čabarkapa,150.0,1550.0,-0.7,1.2,-3.800000,zarkocabarkapa


In [44]:
# merge college data and nba data dataframes
# create temp dfs
temp_df_merged_filter = df_merged_filter
df_career_merge = df_career

# merge on 'name_slug', 'season', and 'team' to keep unique
df_final_data = pd.merge(
    temp_df_merged_filter,
    df_career_merge,
    left_on=['name_slug'],
    right_on=['match_slug'],
    how='left'
)

In [45]:
# drop all players who have no career_bpm
df_final_data_filter = df_final_data.dropna(subset=['career_bpm'])

display(df_final_data_filter)

,athlete_id,name_slug,season,season_label,team_id,team,conference,athlete_source_id,name,position,games,starts,minutes,points,turnovers,fouls,assists,steals,blocks,usage,offensive_rating,defensive_rating,net_rating,porpag,effective_field_goal_pct,true_shooting_pct,assists_turnover_ratio,free_throw_rate,offensive_rebound_pct,fg_pct,fg_attempted,fg_made,two_pt_fg_pct,two_pt_fg_attempted,two_pt_fg_made,three_pt_fg_pct,three_pt_fg_attempted,three_pt_fg_made,ft_pct,ft_attempted,ft_made,rebounds_total,rebounds_defensive,rebounds_offensive,win_shares_total_per40,win_shares_total,win_shares_defensive,win_shares_offensive,hometown_state,hometown_city,date_of_birth,start_season,end_season,age,height_inches,weight_lbs,draft_year,draft_round,overall_pick,source_team_location,Player_Clean,total_games,total_minutes,total_vorp,total_ws,career_bpm,match_slug
0,86485,spencerhawes,2007,20062007,343,Washington,Pac-12,31657,Spencer Hawes,C,31,23,896,461,78,70,60,16,54,25.7,111.3,116.2,-4.9,2.8,53.3,0.568,0.77,27.0,31.0,53.2,363,193,53.3,360,192,33.3,3,1,75.5,98,74,197,136,61,0.121,2.7,0.2,2.5,WA,Seattle,1988-04-28,2007.0,2007.0,19.0,85.0,244.0,2007.0,1.0,10.0,Washington,Spencer Hawes,684.0,15541.0,3.3,22.6,-1.100000,spencerhawes
2,86979,nickyoung,2007,20062007,323,USC,Pac-12,22162,Nick Young,G-F,36,23,1201,630,88,90,48,25,10,26.4,113.1,143.0,-29.9,3.6,57.1,0.614,0.55,35.4,25.4,52.3,444,232,54.3,348,189,44.8,96,43,78.3,157,123,169,126,43,0.083,2.5,-2.8,5.3,CA,Los Angeles,1985-06-01,2005.0,2007.0,22.0,79.0,206.0,2007.0,1.0,16.0,USC,Nick Young,716.0,16382.0,-0.7,18.7,-2.290909,nickyoung
6,86804,arronafflalo,2007,20062007,313,UCLA,Pac-12,22163,Arron Afflalo,G,34,31,1123,577,61,64,67,19,7,25.9,118.0,115.1,2.9,4.2,55.9,0.595,1.10,26.7,15.3,46.1,434,200,54.0,213,115,38.5,221,85,79.3,116,92,98,83,15,0.164,4.6,0.3,4.3,CA,Compton,1985-10-15,2005.0,2007.0,22.0,77.0,210.0,2007.0,1.0,27.0,UCLA,Arron Afflalo,762.0,20829.0,2.4,33.6,-1.863636,arronafflalo
7,85187,acielaw,2007,20062007,293,Texas A&M,Big 12,15169,Acie Law,G,34,22,1153,614,88,83,169,39,1,27.8,118.8,123.8,-5.0,4.8,54.4,0.598,1.92,42.8,7.1,50.0,432,216,51.0,349,178,45.8,83,38,77.8,185,144,113,105,8,0.201,5.8,-0.6,6.4,TX,Dallas,2003-09-19,2004.0,2007.0,4.0,76.0,186.0,2007.0,1.0,11.0,Texas A&M,Acie Law,188.0,2385.0,-1.1,1.6,-3.275000,acielaw
8,85134,kevindurant,2007,20062007,295,Texas,Big 12,31579,Kevin Durant,G-F,35,34,1255,903,99,71,46,66,67,33.1,119.9,99.6,20.3,6.1,53.6,0.594,0.46,39.6,27.2,47.3,647,306,50.5,444,224,40.4,203,82,81.6,256,209,390,284,106,0.255,8.0,2.2,5.8,MD,Suitland,1988-09-29,2007.0,2007.0,19.0,82.0,215.0,2007.0,1.0,2.0,Texas,Kevin Durant,1201.0,44077.0,92.5,186.2,6.472222,kevindurant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
748,15390,isaiahcollier,2024,20232024,323,USC,Pac-12,4683766,Isaiah Collier,G,27,26,804,440,89,64,115,40,6,29.8,105.9,107.8,-1.9,2.6,53.3,0.575,1.29,49.7,21.8,49.0,314,154,54.3,234,127,33.8,80,27,67.3,156,105,78,61,17,0.119,2.4,0.8,1.6,GA,Atlanta,2004-10-08,2024.0,2024.0,20.0,75.0,205.0,2024.0,1.0,29.0,USC,Isaiah Collier,130.0,3356.0,-1.8,0.9,-3.850000,isaiahcollier
751,17024,jonathanmogbo,2024,20232024,259,San Francisco,WCC,5107897,Jonathan Mogbo,F,34,34,984,484,60,89,124,54,28,22.6,131.1,91.0,40.1,4.6,63.6,0.654,2.07,32.1,32.8,63.6,324,206,64.0,322,206,0.0,2,0,69.2,104,72,345,232,113,0.276,6.8,2.5,4.3,FL,West Palm Beach,2001-10-29,2024.0,2024.0,23.0,78.0,217.0,2024.0,2.0,31.0,San Francisco,Jonathan Mogbo,103.0,1535.0,0.1,2.5,-1.300000,jonathanmogbo
752,16084,bubcarrington,2024,20232024,229,Pittsburgh,ACC,4845374,Bub Carrington,G,33,33,1097,456,64,78,136,19,8,23.0,111.9,104.3,7.6,3.0,49.6,0.534,2.13,24.1,9.4,41.2,386,159,51.1,184,94,32.2,202,65,78.5,93,73,171,155,16,0.135,3.7,1.4,2.3,MD,Baltimore,2005-07-21,2024.0,2024.0,19.0,76.0,195

In [46]:
# do some feature engineering
# calculate the precise standard error based on game sample sizes
df_final_data_filter['standard_error'] = np.where(df_final_data_filter['total_games'] > 0, 5.0 / np.sqrt(df_final_data_filter['total_games']), 5.0)

# drop unneeded columns
df_final_data_filter = df_final_data_filter.drop(columns=['match_slug', 'Player_Clean', 'source_team_location'])

In [47]:
# print to a csv
df_final_data_filter.to_csv("college_data_with_nba_metrics.csv", index=False)
print(f"Successfully processed {len(df_final_data_filter)} rows into 'college_data_with_nba_metrics.csv'.")

Successfully processed 488 rows into 'college_data_with_nba_metrics.csv'.


# MERGE 2026 DRAFT CLASS WITH NBA DATA

In [48]:
# load both datasets
df_historical = pd.read_csv("college_data_with_nba_metrics.csv")
df_2026_draft = pd.read_csv("college_data_26 - draft_picks.csv")

# drop unneeded columns
df_historical = df_historical.drop(columns=['athlete_source_id', 'date_of_birth'])

# ensure the columns are in the exact same order before stacking
df_2026_draft = df_2026_draft[df_historical.columns]

# concatenate/Stack them together
df_combined = pd.concat([df_historical, df_2026_draft], ignore_index=True)

# save the master dataset
df_combined.to_csv("nba_draft_model_data.csv", index=False)

print(f"Dataset contains {len(df_combined)} total player rows.")

Dataset contains 544 total player rows.
